# 03. 정상 신호와 스푸핑 신호 비교

## 연구 목적
정상·스푸핑 IQ와 수신기 관측값을 같은 시간축에 정렬해 공격 시작 전후 변화를 관찰합니다.

## 입력
- 정상 IQ: `artifacts/rf_runs/`
- 스푸핑 IQ: `artifacts/spoofing_runs/`
- 선택적 비교표: `artifacts/comparisons/*/comparison.csv`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("저장소 루트 또는 notebooks/에서 Notebook을 실행하세요.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
ARTIFACTS = PROJECT_ROOT / "artifacts"
print("PROJECT_ROOT:", PROJECT_ROOT)


## 중간 확인 1 — 비교 데이터 준비 상태

In [ ]:
from gnss_doppler_lab.research_sequence import latest_run, sequence_status

normal_run = latest_run(ARTIFACTS / 'rf_runs')
status = sequence_status(ARTIFACTS)
print('정상 run:', normal_run)
print('스푸핑 IQ 준비:', status['03_spoofing_comparison'])
print('정렬 비교표 준비:', status['03_comparison_table'])

## 중간 확인 2 — 정상/스푸핑 IQ 스펙트럼 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gnss_doppler_lab.iq_visualization import load_s8_iq

spoof_dir = Path(status['03_spoofing_comparison']['path']) if status['03_spoofing_comparison']['ready'] else None
if spoof_dir is None:
    print('스푸핑 IQ가 아직 없습니다. 이 셀은 데이터 생성 후 자동으로 비교 그래프를 표시합니다.')
else:
    normal = load_s8_iq(normal_run/'gps_l1ca_s8_iq.bin', max_complex_samples=262144)
    spoof = load_s8_iq(spoof_dir/'gps_l1ca_s8_iq.bin', max_complex_samples=262144)
    fig, ax = plt.subplots(figsize=(13, 5))
    for label, x in [('normal', normal), ('spoofing', spoof)]:
        n = min(len(x), 65536); spec = np.fft.fftshift(np.fft.fft(x[:n]*np.hanning(n)))
        db = 20*np.log10(np.maximum(abs(spec),1e-12)); db -= db.max()
        ax.plot(np.linspace(-1.3,1.3,n,endpoint=False), db, lw=.7, label=label)
    ax.set(title='Normal vs spoofing relative spectrum', xlabel='Offset (MHz)', ylabel='dB')
    ax.set_ylim(-100,5); ax.grid(alpha=.25); ax.legend(); plt.show()

## 판정

- [ ] 정상·공격 run이 동일 UTC/위치/trajectory/sample rate 기준으로 비교된다.
- [ ] 공격 시작 시각과 유형·power advantage가 metadata에 기록된다.
- [ ] RF power 차이만으로 train/test label이 누설되지 않는다.
- [ ] PRN별 Doppler, C/N₀, correlator, clock, PVT 변화를 함께 확인한다.
- [ ] 정상과 공격 window가 run 단위로 분리된다.

## 다음 단계
`04_detection_dataset_and_baselines.ipynb`에서 검증된 관측값을 window feature 데이터셋으로 만들고 baseline을 평가합니다.